# H5 Dataset Visualizer for ONEAT Training Data
Interactive visualization of images and labels from H5 training data

In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox, Output, Label
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
# Set the path to your H5 file
H5_PATH = "/home/debian/jean-zay/oneat_training/oneat_kapoorlabs.h5"

# Event names for label interpretation
EVENT_NAMES = ['normal', 'mitosis']

# Box vector labels
BOX_LABELS = ['x', 'y', 'z', 't', 'h', 'w', 'd', 'conf']

In [3]:
# Open H5 file and inspect structure
h5f = h5py.File(H5_PATH, 'r')

print("H5 File Structure:")
print(f"Root keys: {list(h5f.keys())}")
for split in h5f.keys():
    print(f"\n{split}/:")
    for key in h5f[split].keys():
        print(f"  {key}: {h5f[split][key].shape}, dtype={h5f[split][key].dtype}")

H5 File Structure:
Root keys: ['train', 'val']

train/:
  images: (2470, 3, 8, 64, 64), dtype=float32
  labels: (2470, 10), dtype=float32
  segmentations: (2470, 3, 8, 64, 64), dtype=uint16

val/:
  images: (54, 3, 8, 64, 64), dtype=float32
  labels: (54, 10), dtype=float32
  segmentations: (54, 3, 8, 64, 64), dtype=uint16


In [4]:
def parse_yolo_label(label, event_names, box_labels):
    """
    Parse YOLO label format: [x, y, z, t, h, w, d, conf] + [one-hot categories]
    """
    box_vector_len = 8
    
    # Extract box vector
    box_vector = label[:box_vector_len]
    
    # Extract one-hot categories
    categories = label[box_vector_len:]
    class_idx = np.argmax(categories)
    class_name = event_names[class_idx] if class_idx < len(event_names) else f"class_{class_idx}"
    
    # Format box vector string
    box_str = ", ".join([f"{box_labels[i]}={box_vector[i]:.3f}" for i in range(len(box_labels))])
    
    return {
        'class_idx': class_idx,
        'class_name': class_name,
        'class_probs': categories,
        'box_vector': box_vector,
        'box_str': box_str
    }

In [5]:
class H5Visualizer:
    def __init__(self, h5_file, event_names, box_labels):
        self.h5f = h5_file
        self.event_names = event_names
        self.box_labels = box_labels
        self.current_split = 'train'
        
        # Get data references
        self.images = self.h5f[self.current_split]['images']
        self.labels = self.h5f[self.current_split]['labels']
        self.segs = self.h5f[self.current_split].get('segmentations', None)
        
        # Get dimensions
        self.n_samples = self.images.shape[0]
        self.n_time = self.images.shape[1]
        self.n_z = self.images.shape[2]
        
        # Create output widget
        self.output = Output()
        self.label_output = Output()
        
    def switch_split(self, split):
        self.current_split = split
        self.images = self.h5f[split]['images']
        self.labels = self.h5f[split]['labels']
        self.segs = self.h5f[split].get('segmentations', None)
        self.n_samples = self.images.shape[0]
        self.n_time = self.images.shape[1]
        self.n_z = self.images.shape[2]
        
    def visualize(self, sample_idx, t_idx, z_idx, show_seg):
        with self.output:
            clear_output(wait=True)
            
            # Get image slice
            img = self.images[sample_idx, t_idx, z_idx]
            
            if show_seg and self.segs is not None:
                seg = self.segs[sample_idx, t_idx, z_idx]
                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                
                axes[0].imshow(img, cmap='gray')
                axes[0].set_title(f'Raw Image\nSample {sample_idx}, T={t_idx}, Z={z_idx}')
                axes[0].axis('off')
                
                axes[1].imshow(seg, cmap='tab20')
                axes[1].set_title(f'Segmentation\nSample {sample_idx}, T={t_idx}, Z={z_idx}')
                axes[1].axis('off')
            else:
                fig, ax = plt.subplots(1, 1, figsize=(6, 5))
                ax.imshow(img, cmap='gray')
                ax.set_title(f'Raw Image\nSample {sample_idx}, T={t_idx}, Z={z_idx}')
                ax.axis('off')
            
            plt.tight_layout()
            plt.show()
        
        with self.label_output:
            clear_output(wait=True)
            
            # Parse and display label
            label = self.labels[sample_idx]
            parsed = parse_yolo_label(label, self.event_names, self.box_labels)
            
            print("=" * 60)
            print(f"LABEL for Sample {sample_idx}")
            print("=" * 60)
            print(f"Class: {parsed['class_name']} (index: {parsed['class_idx']})")
            print(f"Class probabilities: {parsed['class_probs']}")
            print(f"\nBox Vector:")
            print(f"  {parsed['box_str']}")
            print("=" * 60)
    
    def create_widgets(self):
        # Split selector
        split_dropdown = Dropdown(
            options=list(self.h5f.keys()),
            value=self.current_split,
            description='Split:'
        )
        
        # Sample slider
        sample_slider = IntSlider(
            min=0, max=self.n_samples-1, step=1, value=0,
            description='Sample:',
            continuous_update=False
        )
        
        # Time slider
        t_slider = IntSlider(
            min=0, max=self.n_time-1, step=1, value=self.n_time//2,
            description='Time (T):',
            continuous_update=False
        )
        
        # Z slider
        z_slider = IntSlider(
            min=0, max=self.n_z-1, step=1, value=self.n_z//2,
            description='Z slice:',
            continuous_update=False
        )
        
        # Show segmentation checkbox
        seg_checkbox = widgets.Checkbox(
            value=True,
            description='Show Segmentation',
            disabled=self.segs is None
        )
        
        def on_split_change(change):
            self.switch_split(change['new'])
            sample_slider.max = self.n_samples - 1
            sample_slider.value = min(sample_slider.value, self.n_samples - 1)
            t_slider.max = self.n_time - 1
            t_slider.value = min(t_slider.value, self.n_time - 1)
            z_slider.max = self.n_z - 1
            z_slider.value = min(z_slider.value, self.n_z - 1)
            seg_checkbox.disabled = self.segs is None
            self.visualize(sample_slider.value, t_slider.value, z_slider.value, seg_checkbox.value)
        
        split_dropdown.observe(on_split_change, names='value')
        
        def update(sample_idx, t_idx, z_idx, show_seg):
            self.visualize(sample_idx, t_idx, z_idx, show_seg)
        
        # Create interactive widget
        interactive_widget = widgets.interactive(
            update,
            sample_idx=sample_slider,
            t_idx=t_slider,
            z_idx=z_slider,
            show_seg=seg_checkbox
        )
        
        # Layout
        controls = VBox([
            Label(f'Dataset: {self.n_samples} samples, shape: {self.images.shape}'),
            split_dropdown,
            sample_slider,
            t_slider,
            z_slider,
            seg_checkbox
        ])
        
        display(controls)
        display(self.output)
        display(self.label_output)
        
        # Initial visualization
        self.visualize(0, self.n_time//2, self.n_z//2, True)

In [6]:
# Create and display the visualizer
viz = H5Visualizer(h5f, EVENT_NAMES, BOX_LABELS)
viz.create_widgets()

Output()

Output()

In [7]:
# Optional: Close H5 file when done
# h5f.close()